# 🤖 Retail Sales — Machine Learning (Regression)
**Target:** Predict Sales | **Models:** Linear, Ridge, Lasso, Random Forest

## 1. Setup

In [ ]:
import sys, os
sys.path.append(os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from data_loader import load_csv

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
print('Setup complete ✅')

## 2. Load & Encode Features

In [ ]:
rows = load_csv()
df   = pd.DataFrame(rows)

# Cast numerics
for col in ['Sales', 'Profit', 'Discount', 'Quantity']:
    df[col] = pd.to_numeric(df[col])

# Label encode categorical columns
le = LabelEncoder()
df['Category_enc'] = le.fit_transform(df['Category'])
df['Region_enc']   = le.fit_transform(df['Region'])
df['Segment_enc']  = le.fit_transform(df['Segment'])
df['ShipMode_enc'] = le.fit_transform(df['Ship Mode'])

print('Encoding:')
for col in ['Category', 'Region', 'Segment']:
    orig = df[col].unique()
    enc  = df[f'{col}_enc'].unique()
    print(f'  {col}: {list(orig)} → {list(enc)}')

## 3. Feature Scaling & Train/Test Split

In [ ]:
features = ['Quantity', 'Discount', 'Profit',
            'Category_enc', 'Region_enc', 'Segment_enc', 'ShipMode_enc']

X = df[features]
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

print(f'Train : {X_train.shape[0]} rows')
print(f'Test  : {X_test.shape[0]} rows')
print(f'Features: {features}')

## 4. Train Models & Evaluate

In [ ]:
def evaluate(name, y_test, y_pred):
    mae  = mean_absolute_error(y_test, y_pred)
    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_test, y_pred)
    print(f'\n{name}')
    print(f'  MAE  : {mae:.2f}')
    print(f'  RMSE : {rmse:.2f}')
    print(f'  R²   : {r2:.4f}')
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'y_pred': y_pred}

results = []

# Linear Regression
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
results.append(evaluate('Linear Regression', y_test, lr.predict(X_test_sc)))

# Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_sc, y_train)
results.append(evaluate('Ridge Regression', y_test, ridge.predict(X_test_sc)))

# Lasso
lasso = Lasso(alpha=1.0)
lasso.fit(X_train_sc, y_train)
results.append(evaluate('Lasso Regression', y_test, lasso.predict(X_test_sc)))

# Random Forest (no scaling needed)
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
results.append(evaluate('Random Forest', y_test, rf.predict(X_test)))

## 5. Model Comparison Summary

In [ ]:
summary = pd.DataFrame([{k: v for k, v in r.items() if k != 'y_pred'} for r in results])
summary = summary.set_index('model').round(4)
print(summary.to_string())
print(f'\n✅ Best model: {summary["R2"].idxmax()} (R² = {summary["R2"].max():.4f})')

## 6. Actual vs Predicted Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Actual vs Predicted Sales', fontsize=15, fontweight='bold')
axes = axes.flatten()

for i, r in enumerate(results):
    ax = axes[i]
    ax.scatter(y_test, r['y_pred'], alpha=0.3, s=10, color='#4C72B0')
    mn = min(float(y_test.min()), float(r['y_pred'].min()))
    mx = max(float(y_test.max()), float(r['y_pred'].max()))
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_title(f"{r['model']}\nR² = {r['R2']:.4f}", fontweight='bold')
    ax.set_xlabel('Actual Sales ($)')
    ax.set_ylabel('Predicted Sales ($)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7. Feature Importance — Random Forest

In [ ]:
importances = dict(zip(features, rf.feature_importances_))
sorted_imp  = sorted(importances.items(), key=lambda x: x[1])

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([i[0] for i in sorted_imp], [i[1] for i in sorted_imp],
        color=sns.color_palette('Blues_d', len(sorted_imp)))
ax.set_title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('\nImportance scores:')
for feat, imp in sorted(importances.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(imp * 50)
    print(f'  {feat:<15} {bar} {imp:.4f}')